In [ ]:
from pathlib import Path
import json
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "pinn_swe_reflections").is_dir():
            return candidate
    raise RuntimeError("Cannot find project root. Run this notebook from inside the repository.")


ROOT_DIR = find_project_root()
TRAINING_RESULTS_DIR = Path(
    os.environ.get("PINN_SWE_TRAINING_RESULTS_DIR", ROOT_DIR / "training_results")
).expanduser().resolve()
OUTPUT_DIR = ROOT_DIR / "analysis_outputs" / "saved_run_evaluation"

SAVE_OUTPUTS = True
FIG_DPI = 180

# Either list experiments explicitly, or set AUTO_DISCOVER_EXPERIMENTS = True.
AUTO_DISCOVER_EXPERIMENTS = False
EXPERIMENTS = [
    {
        "name": "baseline",
        "label": "PINN baseline",
        "run_dir": TRAINING_RESULTS_DIR / "baseline",
    },
    # {
    #     "name": "gpinn_w001",
    #     "label": "gPINN w=0.01",
    #     "run_dir": TRAINING_RESULTS_DIR / "gpinn_w001",
    # },
    # {
    #     "name": "rad_k1_c1",
    #     "label": "RAD k=1 c=1",
    #     "run_dir": TRAINING_RESULTS_DIR / "rad_k1_c1",
    # },
    # {
    #     "name": "rar_d_k1_c1",
    #     "label": "RAR-D k=1 c=1",
    #     "run_dir": TRAINING_RESULTS_DIR / "rar_d_k1_c1",
    # },
]

if AUTO_DISCOVER_EXPERIMENTS:
    EXPERIMENTS = [
        {"name": p.name, "label": p.name, "run_dir": p}
        for p in sorted(TRAINING_RESULTS_DIR.iterdir())
        if p.is_dir() and (p / "Hyper_Parameter_Dictionary.json").is_file()
    ]

if SAVE_OUTPUTS:
    (OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT_DIR}")
print(f"Training results: {TRAINING_RESULTS_DIR}")
print(f"Experiments: {[e['name'] for e in EXPERIMENTS]}")

In [ ]:
VARIABLE_SPECS = {
    "eta": {
        "label": "eta / zeta",
        "reference": "exact_solution_h_values.npy",
        "prediction_dimensional": "dimensional_network_output_h_values.npy",
        "prediction_raw": "network_output_h_values.npy",
        "scale_kind": "vertical",
        "time_mesh": "dimensional_zeta_solution_time_mesh_grid.npy",
        "x_mesh": "dimensional_zeta_solution_x_mesh_grid.npy",
        "unit": "m",
    },
    "u": {
        "label": "u",
        "reference": "exact_solution_u_values.npy",
        "prediction_dimensional": "dimensional_network_output_u_values.npy",
        "prediction_raw": "network_output_u_values.npy",
        "scale_kind": "velocity",
        "time_mesh": "dimensional_u_solution_time_mesh_grid.npy",
        "x_mesh": "dimensional_u_solution_x_mesh_grid.npy",
        "unit": "m/s",
    },
}

LOSS_SPECS = {
    "total": "total_MSE_over_training.npy",
    "pde": "MSE_symbolic_functions_over_training.npy",
    "initial": "MSE_initial_conditions_over_training.npy",
    "boundary": "MSE_boundary_conditions_over_training.npy",
    "pde_u": "MSE_symbolic_function_u_over_training.npy",
    "pde_eta": "MSE_symbolic_function_h_over_training.npy",
    "reference_mse_eta": "Relative_L2_Error_h_over_training.npy",
    "reference_mse_u": "Relative_L2_Error_u_over_training.npy",
    "gpinn_gradient": "MSE_gPINN_gradient_over_training.npy",
    "gpinn_base_pde": "MSE_symbolic_function_without_gpinn_over_training.npy",
}

ADAPTIVE_HISTORY_FILES = {
    "rad_mean_residual": "rad_mean_residual_over_training.npy",
    "rad_max_residual": "rad_max_residual_over_training.npy",
    "rad_min_residual": "rad_min_residual_over_training.npy",
    "rar_d_added_points": "rar_d_added_points_over_training.npy",
    "rar_d_pde_dataset_size": "rar_d_pde_dataset_size_over_training.npy",
    "rar_d_mean_residual": "rar_d_mean_residual_over_training.npy",
    "rar_d_max_residual": "rar_d_max_residual_over_training.npy",
    "rar_d_min_residual": "rar_d_min_residual_over_training.npy",
}


def read_json(path, default=None):
    path = Path(path)
    if not path.is_file():
        return default
    return json.loads(path.read_text(encoding="utf-8"))


def load_npy(path, required=True):
    path = Path(path)
    if not path.is_file():
        if required:
            raise FileNotFoundError(path)
        return None
    return np.load(path, allow_pickle=True)


def as_float_array(values):
    return np.asarray(values, dtype=np.float64)


def dimensionalize_prediction(raw, hp, scale_kind):
    raw = as_float_array(raw)
    if not hp.get("non_dimensionalization", False):
        return raw
    if scale_kind == "vertical":
        return raw * float(hp["vertical_length_scale"])
    if scale_kind == "velocity":
        return raw * float(hp["horizontal_length_scale"]) / float(hp["time_scale"])
    raise ValueError(f"Unknown scale kind: {scale_kind}")


def load_prediction(run_dir, hp, spec):
    dimensional = load_npy(run_dir / spec["prediction_dimensional"], required=False)
    if dimensional is not None:
        return as_float_array(dimensional)
    raw = load_npy(run_dir / spec["prediction_raw"], required=True)
    return dimensionalize_prediction(raw, hp, spec["scale_kind"])


def compute_metrics(reference, prediction):
    reference = as_float_array(reference)
    prediction = as_float_array(prediction)
    diff = prediction - reference
    l1_denom = np.sum(np.abs(reference))
    l2_denom = np.linalg.norm(reference.ravel())
    return {
        "relative_l1": np.nan if l1_denom == 0 else np.sum(np.abs(diff)) / l1_denom,
        "relative_l2": np.nan if l2_denom == 0 else np.linalg.norm(diff.ravel()) / l2_denom,
        "mse": np.mean(diff ** 2),
        "rmse": np.sqrt(np.mean(diff ** 2)),
        "mae": np.mean(np.abs(diff)),
        "max_abs": np.max(np.abs(diff)),
        "n_points": int(diff.size),
    }


def history_epochs(values, hp):
    values = np.asarray(values).reshape(-1)
    output_period = int(hp.get("console_output_period", hp.get("output_period", 1)) or 1)
    if len(values) == 0:
        return np.array([], dtype=int)
    epochs = np.arange(len(values), dtype=int) * output_period
    epochs[0] = 0
    max_epoch = hp.get("epochs")
    if max_epoch is not None and len(values) > 1:
        epochs[-1] = min(int(max_epoch), int(epochs[-1]))
    return epochs


def save_figure(fig, stem):
    if not SAVE_OUTPUTS:
        return
    fig.savefig(OUTPUT_DIR / "figures" / f"{stem}.png", dpi=FIG_DPI, bbox_inches="tight")


def safe_name(text):
    return "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in str(text))

In [ ]:
loaded_runs = []
metric_rows = []
loss_rows = []
adaptive_rows = []

for exp in EXPERIMENTS:
    run_dir = Path(exp["run_dir"]).expanduser().resolve()
    if not run_dir.is_dir():
        warnings.warn(f"Skipping missing run directory: {run_dir}")
        continue

    hp = read_json(run_dir / "Hyper_Parameter_Dictionary.json", default={})
    run = {
        "name": exp.get("name", run_dir.name),
        "label": exp.get("label", exp.get("name", run_dir.name)),
        "run_dir": run_dir,
        "hyperparameters": hp,
        "fields": {},
    }

    for variable, spec in VARIABLE_SPECS.items():
        try:
            reference = as_float_array(load_npy(run_dir / spec["reference"], required=True))
            prediction = load_prediction(run_dir, hp, spec)
        except FileNotFoundError as exc:
            warnings.warn(f"{run['name']}: missing {variable} array: {exc}")
            continue

        if reference.shape != prediction.shape:
            raise ValueError(
                f"{run['name']} {variable}: reference shape {reference.shape} != prediction shape {prediction.shape}"
            )

        diff = prediction - reference
        run["fields"][variable] = {
            "reference": reference,
            "prediction": prediction,
            "abs_error": np.abs(diff),
            "time_mesh": load_npy(run_dir / spec["time_mesh"], required=False),
            "x_mesh": load_npy(run_dir / spec["x_mesh"], required=False),
        }

        row = {
            "experiment": run["name"],
            "label": run["label"],
            "variable": variable,
            "run_dir": str(run_dir),
            "model_class": hp.get("model_class"),
            "best_step": hp.get("best_step"),
            "epochs": hp.get("epochs"),
            "computation_time_s": hp.get("computation_time"),
        }
        row.update(compute_metrics(reference, prediction))
        metric_rows.append(row)

    for loss_name, filename in LOSS_SPECS.items():
        values = load_npy(run_dir / filename, required=False)
        if values is None:
            continue
        values = np.asarray(values, dtype=np.float64).reshape(-1)
        epochs = history_epochs(values, hp)
        for epoch, value in zip(epochs, values):
            loss_rows.append({
                "experiment": run["name"],
                "label": run["label"],
                "loss": loss_name,
                "epoch": int(epoch),
                "value": float(value),
            })

    for history_name, filename in ADAPTIVE_HISTORY_FILES.items():
        values = load_npy(run_dir / filename, required=False)
        if values is None:
            continue
        values = np.asarray(values, dtype=np.float64).reshape(-1)
        for index, value in enumerate(values):
            adaptive_rows.append({
                "experiment": run["name"],
                "label": run["label"],
                "history": history_name,
                "index": index,
                "value": float(value),
            })

    loaded_runs.append(run)

if not loaded_runs:
    raise RuntimeError("No experiments were loaded. Check EXPERIMENTS or AUTO_DISCOVER_EXPERIMENTS.")

metrics_df = pd.DataFrame(metric_rows)
loss_df = pd.DataFrame(loss_rows)
adaptive_df = pd.DataFrame(adaptive_rows)

metrics_df

In [ ]:
summary_columns = [
    "experiment",
    "label",
    "variable",
    "relative_l1",
    "relative_l2",
    "mse",
    "rmse",
    "mae",
    "max_abs",
    "best_step",
    "epochs",
    "computation_time_s",
]
summary_df = metrics_df[summary_columns].sort_values(["variable", "relative_l2", "experiment"])

display(summary_df)

pivot_df = summary_df.pivot_table(
    index=["experiment", "label"],
    columns="variable",
    values=["relative_l1", "relative_l2", "rmse", "max_abs"],
)
display(pivot_df)

if SAVE_OUTPUTS:
    tables_dir = OUTPUT_DIR / "tables"
    summary_df.to_csv(tables_dir / "final_metrics.csv", index=False)
    pivot_df.to_csv(tables_dir / "final_metrics_pivot.csv")
    try:
        (tables_dir / "final_metrics.tex").write_text(summary_df.to_latex(index=False), encoding="utf-8")
    except Exception as exc:
        warnings.warn(f"Could not write LaTeX table: {exc}")

In [ ]:
if loss_df.empty:
    warnings.warn("No loss histories were found.")
else:
    display(loss_df.head())
    if SAVE_OUTPUTS:
        loss_df.to_csv(OUTPUT_DIR / "tables" / "loss_histories.csv", index=False)

    selected_losses = [
        "total",
        "pde",
        "initial",
        "boundary",
        "reference_mse_eta",
        "reference_mse_u",
    ]
    selected_losses = [name for name in selected_losses if name in set(loss_df["loss"])]

    ncols = 2
    nrows = int(np.ceil(len(selected_losses) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.6 * nrows), squeeze=False)
    axes_flat = axes.ravel()

    for ax, loss_name in zip(axes_flat, selected_losses):
        subset = loss_df[loss_df["loss"] == loss_name]
        for label, group in subset.groupby("label", sort=False):
            group = group.sort_values("epoch")
            ax.semilogy(group["epoch"], group["value"], marker="o", markersize=3, linewidth=1.5, label=label)
        ax.set_title(loss_name)
        ax.set_xlabel("epoch")
        ax.set_ylabel("saved MSE / loss")
        ax.grid(True, which="both", alpha=0.25)
        ax.legend(fontsize=8)

    for ax in axes_flat[len(selected_losses):]:
        ax.axis("off")

    fig.suptitle("Loss histories by experiment", y=1.01)
    fig.tight_layout()
    save_figure(fig, "loss_history_comparison")
    plt.show()

In [ ]:
metrics_to_plot = ["relative_l2", "relative_l1", "rmse"]
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(5.2 * len(metrics_to_plot), 4.2), squeeze=False)

for ax, metric in zip(axes.ravel(), metrics_to_plot):
    table = summary_df.pivot(index="label", columns="variable", values=metric)
    table.plot(kind="bar", ax=ax, width=0.8)
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.grid(True, axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=30)

fig.tight_layout()
save_figure(fig, "final_metric_bars")
plt.show()

In [ ]:
def plot_field_triptych(run, variable):
    spec = VARIABLE_SPECS[variable]
    field = run["fields"].get(variable)
    if field is None:
        return None

    reference = field["reference"]
    prediction = field["prediction"]
    abs_error = field["abs_error"]
    time_mesh = field["time_mesh"]
    x_mesh = field["x_mesh"]

    if time_mesh is not None and x_mesh is not None:
        x_axis = np.asarray(time_mesh, dtype=np.float64) / 86400.0
        y_axis = np.asarray(x_mesh, dtype=np.float64) / 1000.0
        extent = None
    else:
        x_axis = None
        y_axis = None
        extent = [0, reference.shape[1] - 1, 0, reference.shape[0] - 1]

    vmax = max(np.nanmax(np.abs(reference)), np.nanmax(np.abs(prediction)))
    error_vmax = np.nanmax(abs_error)
    panels = [
        ("reference", reference, "RdBu_r", -vmax, vmax),
        ("prediction", prediction, "RdBu_r", -vmax, vmax),
        ("absolute error", abs_error, "magma", 0.0, error_vmax),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
    for ax, (title, values, cmap, vmin, vmax_panel) in zip(axes, panels):
        if x_axis is not None:
            mesh = ax.pcolormesh(x_axis, y_axis, values, shading="auto", cmap=cmap, vmin=vmin, vmax=vmax_panel, rasterized=True)
            ax.set_xlabel("time, days")
            ax.set_ylabel("x, km")
        else:
            mesh = ax.imshow(values, origin="lower", aspect="auto", extent=extent, cmap=cmap, vmin=vmin, vmax=vmax_panel)
            ax.set_xlabel("time index")
            ax.set_ylabel("space index")
        ax.set_title(title)
        fig.colorbar(mesh, ax=ax, label=spec["unit"])

    fig.suptitle(f"{run['label']} - {spec['label']}", y=1.05)
    save_figure(fig, f"field_triptych_{safe_name(run['name'])}_{variable}")
    return fig


for run in loaded_runs:
    for variable in VARIABLE_SPECS:
        fig = plot_field_triptych(run, variable)
        if fig is not None:
            plt.show()

In [ ]:
if adaptive_df.empty:
    print("No RAD/RAR-D adaptive histories found in the selected runs.")
else:
    if SAVE_OUTPUTS:
        adaptive_df.to_csv(OUTPUT_DIR / "tables" / "adaptive_histories.csv", index=False)

    histories = list(adaptive_df["history"].drop_duplicates())
    ncols = 2
    nrows = int(np.ceil(len(histories) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.5 * nrows), squeeze=False)
    axes_flat = axes.ravel()

    for ax, history in zip(axes_flat, histories):
        subset = adaptive_df[adaptive_df["history"] == history]
        for label, group in subset.groupby("label", sort=False):
            group = group.sort_values("index")
            ax.plot(group["index"], group["value"], marker="o", markersize=3, linewidth=1.5, label=label)
        ax.set_title(history)
        ax.set_xlabel("adaptive update index")
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8)

    for ax in axes_flat[len(histories):]:
        ax.axis("off")

    fig.tight_layout()
    save_figure(fig, "adaptive_histories")
    plt.show()